In [1]:
import pandas as pd
import numpy as np
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_absolute_error, mean_squared_error
# from sklearn.linear_model import LinearRegression
# import xgboost as xgb


In [2]:
sales_data = pd.read_csv("sales_train_evaluation.csv")
# calendar_data = pd.read_csv("calendar.csv")
# prices_data = pd.read_csv("sell_prices.csv")

In [3]:
# Melt the dataset to a long format
data_long = sales_data.melt(id_vars=['store_id', 'dept_id', 'item_id','id','cat_id','state_id'], 
                      var_name='day', 
                      value_name='sales')

# Convert `day` column to numerical format
data_long['day'] = data_long['day'].str.extract('(\d+)').astype(int)

# Sort by store, department, item, and day
data_long = data_long.sort_values(['store_id', 'dept_id', 'item_id','id','cat_id','state_id', 'day'])
del sales_data # delete the original dataframe to free RAM


In [4]:
data_long

,store_id,dept_id,item_id,id,cat_id,state_id,day,sales
1612,CA_1,FOODS_1,FOODS_1_001,FOODS_1_001_CA_1_evaluation,FOODS,CA,1,3
32102,CA_1,FOODS_1,FOODS_1_001,FOODS_1_001_CA_1_evaluation,FOODS,CA,2,0
62592,CA_1,FOODS_1,FOODS_1_001,FOODS_1_001_CA_1_evaluation,FOODS,CA,3,0
93082,CA_1,FOODS_1,FOODS_1_001,FOODS_1_001_CA_1_evaluation,FOODS,CA,4,1
123572,CA_1,FOODS_1,FOODS_1_001,FOODS_1_001_CA_1_evaluation,FOODS,CA,5,4
...,...,...,...,...,...,...,...,...
59057692,WI_3,HOUSEHOLD_2,HOUSEHOLD_2_516,HOUSEHOLD_2_516_WI_3_evaluation,HOUSEHOLD,WI,1937,0
59088182,WI_3,HOUSEHOLD_2,HOUSEHOLD_2_516,HOUSEHOLD_2_516_WI_3_evaluation,HOUSEHOLD,WI,1938,0
59118672,WI_3,HOUSEHOLD_2,HOUSEHOLD_2_516,HOUSEHOLD_2_516_WI_3_evaluation,HOUSEHOLD,WI,1939,0
59149162,WI_3,HOUSEHOLD_2,HOUSEHOLD_2_516,HOUSEHOLD_2_516_WI_3_evaluation,HOUSEHOLD,WI,1940,0


In [ ]:
sales_data_long = sales_data.melt(
    id_vars=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
    var_name="d",
    value_name="sales",
)

In [ ]:
sales_data_long = sales_data_long.merge(
    calendar_data, how="left", left_on="d", right_on="d"
)

In [ ]:
sales_data_long = sales_data_long.merge(
    prices_data, how="left", on=["store_id", "item_id", "wm_yr_wk"]
)

In [ ]:
sales_data_long.fillna(0, inplace=True)


In [ ]:
sales_data_long.head()

In [ ]:
sales_data_long["lag_7"] = sales_data_long.groupby("id")["sales"].shift(7)

In [ ]:
sales_data_long["lag_14"] = sales_data_long.groupby("id")["sales"].shift(14)


In [ ]:
sales_data_long["rolling_mean_7"] = (
    sales_data_long.groupby("id")["sales"].shift(7).rolling(7).mean()
)

In [ ]:
sales_data_long.dropna(inplace=True)


In [ ]:
sales_data_long.head()

In [ ]:
# Select features and target
features = [
    "sales",
    "lag_7",
    "lag_14",
    "rolling_mean_7",
    "snap_CA",
    "snap_TX",
    "snap_WI",
]
target = "sales"

# Train-test split
X = sales_data_long[features]
y = sales_data_long[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# Train XGBoost Model
xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror", n_estimators=100, learning_rate=0.1, max_depth=6
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

# Evaluate models
print("Linear Regression MAE:", mean_absolute_error(y_test, y_pred_lr))
print("XGBoost MAE:", mean_absolute_error(y_test, y_pred_xgb))
print("Linear Regression RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("XGBoost RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_xgb)))
